# Aire y Urgencias — PySpark en Colab

Lee las tablas del proyecto en Parquet y arma el grano fino del análisis:
**establecimiento × día** de urgencias respiratorias y **estación × día** de MP2.5,
para las tres ciudades del estudio.

> **Asociación, nunca causalidad.** Este cuaderno describe cómo covarían dos series
> agregadas. Es un estudio ecológico observacional: ninguna fila dice qué persona
> respiró qué aire.

## Antes de empezar, algo honesto sobre Spark aquí

Colab da **una sola máquina**. Spark corre en modo local: usás su API, su plan de
ejecución y su lectura de Parquet particionado, pero **no hay cluster ni distribución
real**. Para 209 MB de datos, pandas o Polars son más rápidos — el JVM tarda más en
arrancar que lo que tarda la consulta.

Eso no lo hace inútil: sirve para **demostrar el manejo de la herramienta** y para
escribir el código exactamente igual al que correría en un cluster de verdad. Solo
conviene saberlo y no venderlo como otra cosa en la presentación.

## 1 · Entorno

Colab ya trae una JVM y normalmente también PySpark. La celda **detecta** lo que
hay en vez de instalar a ciegas: fijar una versión choca con la imagen del día, y
un `apt-get install` sin `update` previo falla con 404 cuando el espejo ya reemplazó
el paquete.


In [ ]:
# Detectar en vez de suponer: Colab ya trae una JVM y suele traer PySpark.
# Fijar una version a ciegas rompe contra la imagen del dia.
import glob, os, subprocess, sys

jvms = sorted(glob.glob("/usr/lib/jvm/*"))
print("JVM presentes:", jvms or "ninguna")

if not any(os.path.exists(f"{j}/bin/java") for j in jvms):
    # `apt-get update` primero: sin eso el indice queda viejo y la descarga da 404.
    subprocess.run("apt-get -qq update", shell=True, check=True)
    subprocess.run("apt-get -qq install -y default-jre-headless", shell=True, check=True)
    jvms = sorted(glob.glob("/usr/lib/jvm/*"))

casa = next((j for j in jvms if os.path.exists(f"{j}/bin/java")), None)
assert casa, f"Ninguna JVM utilizable en {jvms}"
os.environ["JAVA_HOME"] = casa
print("JAVA_HOME =", casa)
print(subprocess.run([f"{casa}/bin/java", "-version"],
                     capture_output=True, text=True).stderr.strip())

# PySpark: usar el que ya esta. Instalar solo si falta, y sin fijar version.
try:
    import pyspark
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "pyspark"], check=True)
    import pyspark
print("pyspark", pyspark.__version__)


## 2 · Los datos

La carpeta `processed/` pesa **209 MB**. Se sube una vez a Drive y se comparte con
el equipo; nadie vuelve a bajarla.

No se usan credenciales de AWS en el cuaderno. Un notebook se comparte, y una clave
dentro de un notebook compartido es una clave filtrada.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Ajustar si la carpeta quedó en otra ruta de Drive.
BASE = "/content/drive/MyDrive/aire_urgencias/processed"

import os
assert os.path.isdir(BASE), f"No existe {BASE}. Revisá dónde quedó la carpeta en Drive."
print(sorted(os.listdir(BASE)))

## 3 · La sesión de Spark

Colab da ~12 GB de RAM. Se le asignan 6 al driver: en modo local el driver hace todo
el trabajo, y pedirle demasiado deja sin memoria al propio Python.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("aire-urgencias")
         .master("local[*]")
         .config("spark.driver.memory", "6g")
         .config("spark.sql.session.timeZone", "America/Santiago")
         # Menos particiones de salida: el resultado es chico y 200 archivos
         # vacíos solo hacen ruido.
         .config("spark.sql.shuffle.partitions", "16")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print(spark.version)

## 4 · Leer las tablas

Spark reconoce solo el directorio `anio=2018/`, `anio=2019/`… como **particiones**
y agrega la columna `anio` sin que esté escrita en ningún archivo. Es lo mismo que
hace Athena con `MSCK REPAIR TABLE`.

In [ ]:
urg = spark.read.parquet(f"{BASE}/hecho_urgencia")
med = spark.read.parquet(f"{BASE}/hecho_medicion")
est = spark.read.parquet(f"{BASE}/dim_establecimiento")
cau = spark.read.parquet(f"{BASE}/dim_causa")
tie = spark.read.parquet(f"{BASE}/dim_tiempo")
sta = spark.read.parquet(f"{BASE}/dim_estacion")

print(f"hecho_urgencia : {urg.count():,} filas")
print(f"hecho_medicion : {med.count():,} filas")
urg.printSchema()

## 5 · Las tres ciudades

El cruce va por **código de comuna**, nunca por nombre: el DEIS escribe «Coihaique»
y SINCA «Coyhaique», así que un cruce por nombre pierde esa ciudad entera y no lanza
ningún error.

Santiago son las 32 comunas de la Provincia de Santiago más Puente Alto y San
Bernardo. Talagante (13601) queda fuera por decisión del equipo.

In [ ]:
COMUNAS = {
    "santiago": [13101, 13102, 13103, 13104, 13105, 13106, 13107, 13108, 13109,
                 13110, 13111, 13112, 13113, 13114, 13115, 13116, 13117, 13118,
                 13119, 13120, 13121, 13122, 13123, 13124, 13125, 13126, 13127,
                 13128, 13129, 13130, 13131, 13132, 13201, 13401],
    "talcahuano": [8110],
    "coyhaique": [11101],
}

ciudad = F.when(F.col("comuna_codigo").isin(COMUNAS["santiago"]), "santiago")
for c in ("talcahuano", "coyhaique"):
    ciudad = ciudad.when(F.col("comuna_codigo").isin(COMUNAS[c]), c)

est_ciudad = (est.withColumn("ciudad_id", ciudad.otherwise(None))
                 .filter(F.col("ciudad_id").isNotNull())
                 .select("establecimiento_id", "nombre", "comuna", "ciudad_id"))

est_ciudad.groupBy("ciudad_id").count().orderBy("ciudad_id").show()

## 6 · Grano fino de urgencias: establecimiento × día

Las seis causas respiratorias con CIE-10 explícito. Se usan los **detalles**, no el
agregado `causa_id = 2`, para poder mirar neumonía o influenza por separado.

Los ID de causa se toman de `dim_causa`, no de una lista escrita a mano.

In [ ]:
resp = [r.causa_id for r in cau.filter("es_respiratoria_detalle").collect()]
print("causas respiratorias de detalle:", resp)
cau.filter("es_respiratoria_detalle").select("causa_id", "glosa").show(truncate=False)

urg_dia = (urg
    .filter(F.col("causa_id").isin(resp))
    .join(F.broadcast(est_ciudad), "establecimiento_id")
    .groupBy("ciudad_id", "establecimiento_id", "fecha")
    .agg(F.sum("total").alias("urg_resp"),
         F.sum("menores_1").alias("urg_menores_1"),
         F.sum("de_65_y_mas").alias("urg_65_y_mas")))

print(f"{urg_dia.count():,} filas establecimiento x día")
urg_dia.orderBy("fecha").show(5)

### Qué está haciendo Spark por debajo

`explain()` muestra dos optimizaciones que valen la pena en la presentación:

- **PushedFilters** — el filtro por `causa_id` baja hasta el lector de Parquet, así que
  los bloques que no lo cumplen no se leen.
- **BroadcastHashJoin** — `dim_establecimiento` son 818 filas, así que Spark la manda
  entera a cada nodo en vez de barajar los 66 millones.

In [ ]:
urg_dia.explain(mode="formatted")

## 7 · Grano fino de aire: estación × día

Sin promediar entre estaciones. El promedio por ciudad esconde diferencias de hasta
61 µg/m³ entre estaciones de Santiago en la misma semana; **esa decisión se toma en
el análisis, no en la tabla.**

Un día de estación cuenta si tiene al menos 18 de 24 horas — el criterio del 75 %
que está documentado en `docs/calidad/cobertura_horaria_semanal.md`.

In [ ]:
aire_dia = (med
    .filter(F.col("parametro_id") == "mp25")
    .groupBy("ciudad_id", "estacion_id", "fecha")
    .agg(F.avg("valor").alias("mp25"),
         F.max("valor").alias("mp25_max_hora"),
         F.count("valor").alias("horas"))
    .filter(F.col("horas") >= 18))

print(f"{aire_dia.count():,} filas estación x día válidas")
aire_dia.orderBy(F.desc("mp25")).show(5)

## 8 · Bajar a pandas — el resultado, no la fuente

Aquí está el patrón que usa todo el mundo: la agregación pesada corre en Spark, y a
pandas baja solo el resultado, que es chico. Iterar sobre 48 mil filas es instantáneo;
sobre 66 millones, no.

**Nunca `.toPandas()` sobre `urg` o `med` sin filtrar.** `hecho_urgencia` sin comprimir
son 4,5 GB.

In [ ]:
aire = aire_dia.toPandas()
urgs = urg_dia.toPandas()
print(f"aire {aire.shape} · urgencias {urgs.shape}")

# Guardar el grano fino en Drive para no recalcularlo en cada sesión.
import os
SALIDA = "/content/drive/MyDrive/aire_urgencias/trabajo"
os.makedirs(SALIDA, exist_ok=True)
aire.to_parquet(f"{SALIDA}/aire_estacion_dia.parquet", index=False)
urgs.to_parquet(f"{SALIDA}/urgencias_establecimiento_dia.parquet", index=False)
print("guardado en", SALIDA)

## 9 · Un primer vistazo descriptivo

Describe el material cargado. **No es un resultado**: no hay control por temperatura,
estacionalidad ni pandemia, y la correspondencia visual entre dos series no es
evidencia de asociación.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

a = aire.copy(); a["fecha"] = pd.to_datetime(a.fecha)
u = urgs.copy(); u["fecha"] = pd.to_datetime(u.fecha)

fig, ejes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for eje, ciudad in zip(ejes, ["santiago", "talcahuano", "coyhaique"]):
    ai = (a[a.ciudad_id == ciudad].groupby("fecha").mp25.mean()
          .rolling(7, min_periods=4).mean())
    ui = (u[u.ciudad_id == ciudad].groupby("fecha").urg_resp.sum()
          .rolling(7, min_periods=4).mean())
    eje.plot(ai.index, ai.values, lw=.9, color="#8F5C0C", label="MP2.5 µg/m³")
    eje.set_ylabel("MP2.5", color="#8F5C0C")
    otro = eje.twinx()
    otro.plot(ui.index, ui.values, lw=.9, color="#20614F", label="urgencias resp.")
    otro.set_ylabel("urgencias", color="#20614F")
    eje.set_title(ciudad.capitalize(), loc="left", fontsize=11)
fig.suptitle("MP2.5 y urgencias respiratorias, media móvil de 7 días — descriptivo",
             fontsize=12)
fig.tight_layout()
plt.show()

## 10 · Cerrar la sesión

Colab corta el entorno solo, pero conviene liberar la JVM si se sigue trabajando en
pandas en el mismo cuaderno.

In [ ]:
spark.stop()

---

## Lo que sigue, y no está aquí

El puente **establecimiento × estación con la distancia**, que es la pieza que permite
emparejar cada establecimiento con su sensor más cercano en vez de usar un promedio
de ciudad. Las coordenadas están en `deis_establecimientos_<año>.xlsx` (latitud y
longitud en grados decimales) y en `dim_estacion`.

Medido: **122 de 131 establecimientos tienen coordenada**; a 2 km hay sensor para 44
de ellos (36 %), a 5 km para 107 (88 %). Conviene guardar la distancia como columna y
dejar el corte de radio para la consulta.

Y antes de agregar a semana, cerrar la decisión **diario contra semanal**: el DEIS
publica diario y SINCA horario, así que el grano fino todavía está disponible.